# 04 · Warp specialization

In notebooks 01–03, every warp did every job. `fp16_gemm_2` gives each warp group one exclusive role: one loads, one computes, one drains. The TMA warp can fetch tile N+1 while the MMA warp computes tile N — real task parallelism instead of one warp alternating.

IKET captures each warp group independently, so the roles show up as separate concurrent bands. This is the notebook where the trace stops being a timeline and starts being a picture of the design.

## 1. The question

Three roles, no warp doing two jobs:

| Warp | Job | Its ranges |
|---|---|---|
| **TMA** | All A/B tile loads | `prod_acquire`, `tma_issue` per K-tile |
| **MMA** | All tensor-core MMAs | `cons_wait`, `mma_issue`, `cons_release` |
| **Epilogue** | Drain and write out | `acc_wait` → `t2r` → `r2s` → `tma_store` |

**The question:** do the roles genuinely overlap, and where does the win actually come from?

### Setup

Every notebook starts here. If this machine cannot profile, the collection cells below print their command instead of failing, and the notebook still reads end to end.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent / "tools"))
import check_env, nbhelp, trace_summary as ts

env = check_env.report()

Environment
----------------------------------------------------------
  GPU           NVIDIA B300 SXM6 AC  [SM103 (Blackwell)]
  Driver        610.57.04
  cutlass-dsl   4.7.1
  iket API      yes
  run-iket      yes
----------------------------------------------------------
  Live profiling: AVAILABLE


## 2. The instrumentation

This kernel uses the `_marks` style, which is the one to copy. Async *issues* (`tma_issue`, `mma_issue`, `cons_release`) are `iket.mark` — they are instants, not durations. Only genuine waits and phases are ranges.

That choice halves the event count versus making everything a range, which matters more than it sounds: see notebook 05, where the same decision is the difference between a full trace and an empty one.

In [2]:
nbhelp.show_iket_calls(
    nbhelp.kernel("instrumented", "fp16_gemm_2_iket_detailed_marks.py"),
    limit=16,
)

fp16_gemm_2_iket_detailed_marks.py: 22 IKET call sites (showing first 16)
------------------------------------------------------------------------
>  121      cute.experimental.iket.range_push("prologue")                       # prologue
       ...
>  309      cute.experimental.iket.range_pop()                                  # prologue
       ...
>  320              cute.experimental.iket.range_push("tma_iter")               # tma_iter
       ...
>  322              cute.experimental.iket.range_push("prod_acquire")           # prod_acquire
       ...
>  324              cute.experimental.iket.range_pop()                          # prod_acquire
       ...
>  327              cute.experimental.iket.mark("tma_issue")  # async op -> mark (no artificial range)
       ...
>  342              cute.experimental.iket.range_pop()                          # tma_iter
       ...
>  361                  cute.experimental.iket.range_push("mma_iter")           # mma_iter
       ...
>  363           

## 3. Run it

In [3]:
OUT = "out/nb04_gemm2"
cmd = (
    f"run-iket --output-dir {OUT} --clobber profile --postprocess all -- "
    f"python {nbhelp.kernel("instrumented", "fp16_gemm_2_iket_detailed_marks.py")} --mnk 4096,4096,512"
)
check_env.run_or_show(cmd, env, est="about 1 minute")

$ run-iket --output-dir out/nb04_gemm2 --clobber profile --postprocess all -- python /path/to/cutedsl-iket-tutorial-notebooks/kernels/instrumented/fp16_gemm_2_iket_detailed_marks.py --mnk 4096,4096,512

[run-iket] ============= Dry run application first to collect some info... =============


Running Blackwell fp16 GEMM example 2 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS
[run-iket] Auto-computed context buffer size: 0.00 GB (1,187,020 bytes)
[run-iket] ============= Run application for real profiling... =============


Running Blackwell fp16 GEMM example 2 with:
  mnk:       [4096, 4096, 512]
  tolerance: 0.1

PASS


[run-iket] Dumped perfetto trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/iket_pid_0x391e49.pftrace
[run-iket] Dumped json trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/iket_pid_0x391e49.trace.json


[run-iket] Dumped compressed trace to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/iket_pid_0x391e49.pftrace.gz
[run-iket] Dumped perfetto HTML viewer to /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/iket_pid_0x391e49.html
[run-iket] To view: cd /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2 && python3 -m http.server 8080 , then open http://localhost:8080/iket_pid_0x391e49.html
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/tracker
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/gen-config
[run-iket] Removing intermediate output directory: /path/to/cutedsl-iket-tutorial-notebooks/notebooks/out/nb04_gemm2/iket


0

## 4. The trace

Distinct role bands, running at the same time. This is the figure that makes warp specialization legible:

In [4]:
nbhelp.figure("nb04/gemm2_warp_roles_overlap_a.png", "TMA, MMA and epilogue warps overlapping in time — leader CTA.")

In [5]:
nbhelp.figure("nb04/gemm2_warp_roles_overlap_b.png", "The same, non-leader CTA.")

In [6]:
try:
    trace = ts.load(ts.find_trace("out/nb04_gemm2"))
    ts.print_summary(ts.summarize(trace), top=10)
except FileNotFoundError:
    print("No local trace here — the reference numbers below were measured on a NVIDIA B300.")

kernel_cutlass_kernel_TiledMMA_ThrLayoutVMNK21111000_PermutationMNK____MMAAtom_ThrID21_ShapeMNK25625616_TVLayoutA2128161281256_TVLayoutB2128161281256_TVLayoutC21282561281256_cutlasscutenv_0   grid 32x16x1  block 192x1x1  warps traced 3072
warp-lifetime span: 23.5 us

range                      count      median         p90       total
--------------------------------------------------------------------
acc_wait                    2048      2.752u      2.944u    5667.90u
epi_iter                   16384      0.128u      0.160u    2139.39u
prologue                    3072      0.288u      0.320u     911.17u
tma_iter                    4096      0.096u      0.352u     648.29u
prod_acquire                4096      0.064u      0.320u     490.11u
r2s                        16384      0.032u      0.032u     425.38u
mma_iter                    2048      0.160u      0.352u     406.37u
cons_wait                   2048      0.096u      0.288u     265.73u
t2r                        16384      0.00

## 5. The finding

Measured on a NVIDIA B300 at `mnk=4096,4096,512` — 3,072 warps, 23.84 µs span:

| Range | Median | Count | Reading |
|---|---|---|---|
| `acc_wait` | 2.752 µs | 2,048 | Epilogue idling in the mainloop's shadow — **by design** |
| `tma_iter` | 0.096 µs | 4,096 | Producer loading tile N+1 |
| `epi_iter` | 0.128 µs | 16,384 | One epilogue subtile |
| `prologue` | 0.288 µs | 3,072 | Uniform across all warps |


**The overlap is real.** `tma_iter` at 0.096 µs runs concurrently with MMA work rather than before it. The producer and consumer advance in near-lockstep.

**`acc_wait` is the trap.** At 2.752 µs it is by far the largest range, and it is not a bottleneck — it is the epilogue warp correctly waiting while the mainloop runs. A tool that ranked ranges by duration and called the top one the problem would send you optimising a wait that *should* be long. Interpreting a wait requires knowing which role is waiting and what for.

**Where gemm 2 actually beats gemm 1** is not the steady state — the per-iteration numbers are close. It is the edges: getting into the pipeline (first load at 0.8 µs versus 3.2–3.7 µs) and getting out through a TMA-store epilogue.

In [7]:
nbhelp.figure("nb04/gemm1_vs_gemm2_entry_exit.png", "Where gemm 2 wins: entry and exit, not the steady state.")

**Next:** [05 · Persistent and CLC scheduling](05_persistent_scheduling.ipynb)

---
*Measured on a single NVIDIA B300, public `nvidia-cutlass-dsl` 4.7.1. Your absolute numbers will differ; the shapes in the trace are the transferable part.*